# Make all reviewed assistant targets
One annotated page per sample. Unreviewed pages are reported, never silently converted to UNKNOWN.

No models. Export both fixed splits after labeling.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "project.json").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pipeline_helpers as P
import json
from PIL import Image, ImageDraw, ImageFont
ROOT = P.ROOT
MODELS = P.read_json(ROOT / "models.json")
SPLITS = ["train", "val"]  # Or ["train"] / ["val"].
FORCE_RERUN = False
items = P.pages(SPLITS)
print("Pages:", len(items))
import importlib
import target_helpers
importlib.reload(target_helpers)
from target_helpers import export_page
FORCE_REEXPORT = False


Reuse unchanged exports. Inspect the skipped-page report before training.

In [ ]:
exported, skipped = [], []
for item in items:
    try:
        path = export_page(item, force=FORCE_REEXPORT)
        exported.append(str(path))
    except (FileNotFoundError, ValueError) as error:
        skipped.append({**item, "reason":str(error)})
        print("Needs review:", item["sequence_id"], item["page_number"], error)
P.write_json(ROOT / "data/training/export_report.json", {"exported":exported, "skipped":skipped})
print("Exported:", len(exported), "Needs review:", len(skipped))


## View saved results
These cells only read saved results. They do not run models or change cached files. Training shows targets and run-wide loss history; validation shows saved predictions.

In [ ]:
from pathlib import Path
import sys, importlib
view_root = Path.cwd().resolve()
if not (view_root / "project.json").is_file():
    view_root = view_root.parent
sys.path.insert(0, str(view_root))
import result_viewer as V
importlib.reload(V)
VIEW_STAGE = 'target'
VIEW_SPLIT = "train"  # Or "val".
VIEW_RUN = None


### 1. View a random page
Rerun this cell to choose another page with a saved result in this split.

In [ ]:
viewed_page = V.show_random(VIEW_STAGE, split=VIEW_SPLIT, run_folder=VIEW_RUN)


### 2. View all pages of a sequence
Enter a sequence ID, or leave it empty to use the random page’s sequence. Missing stage results are reported for each page.

In [ ]:
VIEW_SEQUENCE_ID = ""  # Example: "seq_71033fec4de494fc"
sequence_to_view = VIEW_SEQUENCE_ID or (viewed_page["sequence_id"] if viewed_page else "")
if sequence_to_view:
    V.show_sequence(sequence_to_view, VIEW_STAGE, split=VIEW_SPLIT, run_folder=VIEW_RUN)
else:
    print("Set VIEW_SEQUENCE_ID to a sequence in VIEW_SPLIT.")
